# pylogtracer 0.2.0 — Try It Notebook

Run the cells top to bottom. **Sections 1–4 are fully offline** (no LLM needed).
**Section 5 uses Ollama** — start it and pull a model first (`ollama pull qwen2.5:3b`).

Everything runs against a sample log this notebook writes for you, so it works
from any directory.


## 0. Setup

In [ ]:
# If pylogtracer isn't installed yet, uncomment one of these:
# %pip install pylogtracer            # from PyPI
# %pip install -e .                   # from a local checkout

import logging
import pylogtracer
from pylogtracer import LogTracer

# Show pylogtracer's own progress logs; quiet the HTTP libraries.
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
for noisy in ("httpx", "httpcore", "urllib3"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

print("pylogtracer version:", pylogtracer.__version__)

## 1. Write a sample log
A realistic app log with 5 incidents, tracebacks, and INFO/WARN/CRITICAL lines.

In [ ]:
SAMPLE = """\
2026-06-19 08:00:01 INFO  [System] Application starting up, version 2.3.1
2026-06-19 08:00:03 INFO  [Database] Connection pool initialized at db-primary:5432
2026-06-19 09:15:24 ERROR [Payment] Payment gateway timeout after 30s - INC5000001
Traceback (most recent call last):
  File "/app/services/payment.py", line 88, in charge
    raise TimeoutError("Connection to payment-gateway timed out")
TimeoutError: Connection to payment-gateway timed out
2026-06-19 09:15:25 WARNING [Order] Order marked pending due to payment failure - INC5000001
2026-06-19 10:02:12 ERROR [Auth] Authentication failed: invalid credentials for admin@corp.com - INC5000002
2026-06-19 10:02:25 ERROR [Auth] Authentication failed: invalid credentials for admin@corp.com - INC5000002
2026-06-19 10:02:26 CRITICAL [Auth] Account locked after 3 failed attempts - INC5000002
2026-06-19 11:30:47 ERROR [Database] Unable to connect to replica db-replica-2:5432 - INC5000003
ConnectionError: [Errno 111] Connection refused
2026-06-19 13:45:11 ERROR [Reports] Failed to generate report: division by zero - INC5000004
ZeroDivisionError: division by zero
2026-06-19 14:20:34 ERROR [Worker] Out of memory: unable to allocate 512MB - INC5000005
MemoryError: Unable to allocate array
2026-06-19 16:00:00 INFO  [System] GET /api/error-rates 200 ok
"""

with open("demo_app.log", "w", encoding="utf-8") as f:
    f.write(SAMPLE)
print("wrote demo_app.log")

## 2. Library mode (no LLM)

In [ ]:
tracer = LogTracer("demo_app.log")   # offline: regex/pattern classification

print("SUMMARY:")
print(tracer.summary())
print("\nFREQUENCY:")
print(tracer.error_frequency())
print("\nHEALTH:")
print(tracer.health_check())

In [ ]:
print("LAST INCIDENT:")
for e in tracer.last_incident():
    print(" ", e["timestamp"], e["error_type"], "|", e["primary_error"][:60])

print("\nINCIDENT DURATION:", tracer.incident_duration())

print("\nERRORS 10:00-12:00:")
for e in tracer.errors_in_range("2026-06-19 10:00:00", "2026-06-19 12:00:00"):
    print(" ", e["timestamp"], e["error_type"])

In [ ]:
print("SEARCH INC5000002:")
for line in tracer.search("INC5000002")["entries"]:
    print(" ", line.splitlines()[0])

print("\nGET RELATED LOGS INC5000003:")
grl = tracer.get_related_logs("INC5000003")
print("  found:", grl["found"], "| in_cluster:", grl["total_in_cluster"],
      "| has_error_cluster:", grl["has_error_cluster"])

## 3. New in 0.2.0 — robustness

### 3a. Level-aware error detection
The last line `INFO ... /api/error-rates` contains the substring *error* but is
NOT an error. `level_aware=True` reads the real level and excludes it.


In [ ]:
default = LogTracer("demo_app.log").summary()["total_errors"]
aware = LogTracer("demo_app.log", level_aware=True).summary()["total_errors"]
print("default (substring) total_errors :", default)
print("level_aware total_errors         :", aware, " <- the INFO false positive is dropped")

### 3b. JSON-lines logs

In [ ]:
import json
records = [
    {"timestamp": "2026-06-19 10:00:00", "level": "INFO",  "message": "service started"},
    {"timestamp": "2026-06-19 10:00:05", "level": "ERROR", "message": "db connection refused"},
    {"timestamp": "2026-06-19 10:00:06", "level": "ERROR", "message": "retry attempt failed"},
    {"timestamp": "2026-06-19 11:00:00", "level": "WARNING", "message": "disk almost full"},
]
with open("demo_app.jsonl", "w", encoding="utf-8") as f:
    f.write("\n".join(json.dumps(r) for r in records) + "\n")

j = LogTracer("demo_app.jsonl", log_format="json", level_aware=True)
print("JSON-lines total_errors:", j.summary()["total_errors"], "(2 ERROR; WARNING excluded)")

### 3c. gzip + bounded/tail reads (huge logs don't OOM)

In [ ]:
import gzip
with gzip.open("demo_app.log.gz", "wt", encoding="utf-8") as f:
    f.write(SAMPLE)
print("gzip total_errors:", LogTracer("demo_app.log.gz").summary()["total_errors"])

# Bounded read: only the most recent lines are loaded into memory.
tail = LogTracer("demo_app.log", max_lines=3)
print("tail(max_lines=3) total_entries:", tail.summary()["total_entries"])

## 4. New in 0.2.0 — trust

### Configurable PII / secret redaction
Redaction is applied only to text sent to the LLM. It is **auto-on for cloud
providers** and **off for local Ollama** (nothing leaves the machine).


In [ ]:
from pylogtracer.utils.redaction import redact

raw = "login from 10.0.0.5 by admin@corp.com using Bearer abc.def.ghi key sk-ABCDEFGHIJ1234567890"
print("before:", raw)
print("after :", redact(raw))

# Auto policy:
ollama = LogTracer("demo_app.log", llm_config={"provider": "ollama", "model": "x",
                                               "base_url": "http://localhost:11434"})
openai = LogTracer("demo_app.log", llm_config={"provider": "openai", "model": "gpt-4o-mini",
                                               "api_key": "sk-test"})
print("\nredact for ollama (local):", ollama.redact)
print("redact for openai (cloud):", openai.redact)

## 5. Agent mode (needs Ollama)

Start Ollama and `ollama pull qwen2.5:3b`. The cell is wrapped in try/except so
it won't break the notebook if Ollama isn't running.

Tip: `qwen2.5:7b` gives noticeably better agent answers than `3b`.


In [ ]:
CFG = {"provider": "ollama", "model": "qwen2.5:3b", "base_url": "http://localhost:11434"}

try:
    t = LogTracer("demo_app.log", llm_config=CFG, cache_path="demo_keywords.json")

    print("LLM-classified frequency:")
    print(t.error_frequency())            # unknown types now resolved by the LLM
    print("\nlearned keywords:", list(t._classifier.get_keyword_store().keys()))

    print("\nASK: show all logs for INC5000002")
    print(t.ask("show all logs for INC5000002"))
except Exception as e:
    print("Ollama not available / model error (this is fine to skip):", e)

In [ ]:
# Re-running with the same cache_path reuses learned keywords for free.
# (Run the cell above once, then this — fewer/zero LLM calls for known types.)
try:
    t2 = LogTracer("demo_app.log", llm_config=CFG, cache_path="demo_keywords.json")
    print("ROOT CAUSE of the last incident:")
    rc = t2.root_cause_analysis()
    print("  root_cause :", rc.get("root_cause"))
    print("  suggested  :", rc.get("suggested_fix"))
except Exception as e:
    print("Ollama not available (skip):", e)

## 6. The CLI
Installing the package also installs a `pylogtracer` command.

In [ ]:
import subprocess
print(subprocess.run(["pylogtracer", "demo_app.log", "--summary", "--json"],
                     capture_output=True, text=True).stdout)